# Introduction
Artificial intelligence and machine learning are now commonly used in healthcare, particularly to support tasks such as disease screening, diagnosis, and risk prediction (stratification). These technologies can analyse large datasets far more quickly than a human clinician, which is especially valuable in healthcare systems already under significant pressure. In practice, this can help clinicians identify patterns that might otherwise be missed and support faster decision-making.

However, the use of AI in medical settings raises ethical concerns. Decisions made or influenced by algorithms raise important ethical questions, particularly around accountability, transparency, and patient trust. As AI systems become more embedded in clinical workflows, these issues become increasingly difficult to ignore.

A significant ethical challenge associated with medical AI is bias. Machine learning models are trained using historical data, and if that data contains inequalities or imbalances, the resulting models may reproduce those patterns. In a health care setting, this can lead to unequal performance across protected characteristics, such as error rates between gender, age and ethnicity. These disparities can contribute towards a higher rate of false negatives for certain populations, potentially delaying diagnosis and treatment. 



In [2]:
import pandas as pd



In [3]:
data_frame = pd.read_csv('thyroid_cancer_risk_data.csv')

data_frame.head()
data_frame.shape
print(data_frame.dtypes.to_frame("dtype"))

missing_count = data_frame.isna().sum()
print(missing_count)

duplicates = data_frame.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")



                       dtype
Patient_ID             int64
Age                    int64
Gender                object
Country               object
Ethnicity             object
Family_History        object
Radiation_Exposure    object
Iodine_Deficiency     object
Smoking               object
Obesity               object
Diabetes              object
TSH_Level            float64
T3_Level             float64
T4_Level             float64
Nodule_Size          float64
Thyroid_Cancer_Risk   object
Diagnosis             object
Patient_ID             0
Age                    0
Gender                 0
Country                0
Ethnicity              0
Family_History         0
Radiation_Exposure     0
Iodine_Deficiency      0
Smoking                0
Obesity                0
Diabetes               0
TSH_Level              0
T3_Level               0
T4_Level               0
Nodule_Size            0
Thyroid_Cancer_Risk    0
Diagnosis              0
dtype: int64
Number of duplicate rows: 0


In [4]:
data_frame.drop(columns=["Patient_ID"])

for col in data_frame.select_dtypes(include="object").columns:
    data_frame[col] = data_frame[col].str.strip().str.lower()
    
data_frame["target"] = data_frame["Diagnosis"].map({
    "benign": 0,
    "malignant": 1
})

print(data_frame["target"].value_counts())

data_frame["gender"] = data_frame["Gender"].replace({
    "m": "male",
    "f": "female"  
})

print("Gender distribution:")
print(data_frame["gender"].value_counts())



target
0    163196
1     49495
Name: count, dtype: int64
Gender distribution:
gender
female    127527
male       85164
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X = data_frame.drop(columns=["Diagnosis", "target"])
y = data_frame["target"]

catagorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Catarogical columns: {catagorical_cols}")
print(f"Numerical columns: {numerical_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(f"Training set size: {X_train.shape}, Test set size: {X_test.shape}")
print(f"{y_train.value_counts(normalize=True)}")

print(f"{y_test.value_counts(normalize=True)}")

      

Catarogical columns: ['Gender', 'Country', 'Ethnicity', 'Family_History', 'Radiation_Exposure', 'Iodine_Deficiency', 'Smoking', 'Obesity', 'Diabetes', 'Thyroid_Cancer_Risk', 'gender']
Numerical columns: ['Patient_ID', 'Age', 'TSH_Level', 'T3_Level', 'T4_Level', 'Nodule_Size']
Training set size: (159518, 17), Test set size: (53173, 17)
target
0    0.767293
1    0.232707
Name: proportion, dtype: float64
target
0    0.767288
1    0.232712
Name: proportion, dtype: float64
